In [0]:
from pathlib import Path
from pyspark.sql.functions import current_timestamp, lit

catalogo = "will"
esquema = "i30"

def sql_texto(texto):
    return "'" + texto.replace("'", "''") + "'"

comentarios_silver = {
    "silver_viagens": {
        "tabela": "Camada Silver contendo o histórico limpo e tratado das viagens diárias do veículo, com conversão de tipos e cálculo de quilometragem percorrida.",
        "colunas": {
            "DATA": "Data do registro da viagem. Regra de negócio: Convertida para o tipo Date real (AAAA-MM-DD) a partir do formato original em texto.",
            "KM_INICIAL": "Quilometragem do odômetro no início do percurso, em km. Regra de negócio: Tipada como inteiro (Integer) para garantir consistência matemática.",
            "KM_FINAL": "Quilometragem do odômetro no final do percurso, em km. Regra de negócio: Tipada como inteiro (Integer) para fechamento de rota.",
            "KM_RODADAS": "Quilometragem total rodada no dia, em km. Regra de negócio: Calculada subtraindo KM_INICIAL de KM_FINAL quando ambos os valores estão disponíveis (KM_FINAL - KM_INICIAL).",
            "PAINEL_L_100KM": "Consumo médio indicado pelo painel do veículo, em litros por 100 km. Regra de negócio: Tipada como decimal/double para preservar a precisão das casas decimais informadas no painel.",
            "VELOC_MEDIA_KM_H": "Velocidade média desenvolvida no percurso, em km/h. Regra de negócio: Tipada como inteiro para métricas de desempenho da viagem.",
            "_source_file": "Nome do arquivo CSV de origem. Regra de negócio: Metadado herdado da ingestão Bronze para fins de rastreabilidade.",
            "_processed_at": "Data e hora do processamento da tabela na camada Silver. Regra de negócio: Preenchido automaticamente com o timestamp atual para auditoria do pipeline."
        },
    },
    "silver_caronas": {
        "tabela": "Camada Silver contendo o controle consolidado de passageiros transportados e ganhos financeiros obtidos com caronas.",
        "colunas": {
            "DATA": "Data do registro das caronas. Regra de negócio: Padronizada para o tipo Date real (AAAA-MM-DD).",
            "PASSAG_MANHA": "Quantidade de passageiros no período da manhã. Regra de negócio: Tipada como inteiro (Integer).",
            "GANHO_MANHA_REAIS": "Valor financeiro arrecadado com caronas pela manhã, em reais (R$). Regra de negócio: Tipada como double para cálculos financeiros.",
            "PASSAG_NOITE": "Quantidade de passageiros no período da noite. Regra de negócio: Tipada como inteiro (Integer).",
            "GANHO_NOITE_REAIS": "Valor financeiro arrecadado com caronas à noite, em reais (R$). Regra de negócio: Tipada como double para cálculos financeiros.",
            "TOTAL_PASSAGEIROS": "Soma total de passageiros transportados no dia. Regra de negócio: Enriquecimento calculado somando os passageiros da manhã e da noite (PASSAG_MANHA + PASSAG_NOITE), tratando nulos como zero.",
            "TOTAL_GANHO_REAIS": "Soma total dos ganhos financeiros obtidos no dia. Regra de negócio: Enriquecimento calculado somando os ganhos da manhã e da noite (GANHO_MANHA_REAIS + GANHO_NOITE_REAIS).",
            "_source_file": "Nome do arquivo CSV de origem. Regra de negócio: Metadado de rastreabilidade herdado da Bronze.",
            "_processed_at": "Data e hora do processamento na camada Silver. Regra de negócio: Timestamp automático de auditoria."
        },
    },
    "silver_abastecimento": {
        "tabela": "Camada Silver contendo o histórico refinado dos abastecimentos de combustível, com custos unitários e volumetria calculada.",
        "colunas": {
            "DATA": "Data em que foi realizado o abastecimento. Regra de negócio: Padronizada para o tipo Date real (AAAA-MM-DD).",
            "POSTO_PRECO_LITRO": "Preço cobrado por litro de combustível no posto, em reais por litro (R$/L). Regra de negócio: Tipada como double.",
            "POSTO_VALOR_TOTAL": "Valor total pago no abastecimento, em reais (R$). Regra de negócio: Tipada como double para controle financeiro de custos.",
            "LITROS_ABASTECIDOS": "Quantidade de combustível colocada no tanque, em litros. Regra de negócio: Enriquecimento calculado dividindo o valor total pago pelo preço do litro (POSTO_VALOR_TOTAL / POSTO_PRECO_LITRO) quando não informado diretamente.",
            "_source_file": "Nome do arquivo CSV de origem. Regra de negócio: Metadado de rastreabilidade herdado da Bronze.",
            "_processed_at": "Data e hora do processamento na camada Silver. Regra de negócio: Timestamp automático de auditoria."
        },
    },
}

for tabela, definicao in comentarios_silver.items():
    tabela_completa = f"`{catalogo}`.`{esquema}`.`{tabela}`"

    # Aplica comentário na tabela
    spark.sql(
        f"COMMENT ON TABLE {tabela_completa} "
        f"IS {sql_texto(definicao['tabela'])}"
    )

    # Aplica comentários nas colunas
    for coluna, comentario in definicao["colunas"].items():
        coluna_sql = f"`{coluna.replace('`', '``')}`"

        spark.sql(
            f"COMMENT ON COLUMN {tabela_completa}.{coluna_sql} "
            f"IS {sql_texto(comentario)}"
        )

    print(f"Dicionário Silver aplicado com sucesso: {catalogo}.{esquema}.{tabela}")